# Bot Question Tracking - API Augmentation (011a)

**Date:** 2026-03-21  
**Input:** `products/Bot_Question_Tracking_2026-03-21_v01.csv` (182 questions from HTML)  
**Output:** `products/Bot_Question_Tracking_API_2026-03-21_v01.csv`  

Adds API-sourced fields: my_forecast, open_date, resolution_date, resolution_value, question_type.  
Requires `METACULUS_BOT_API_TOKEN` env var for authenticated forecast data.

In [1]:
# Config + Load HTML data
import requests
import pandas as pd
import time
import json
import os
from pathlib import Path

INPUT_FILE = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products/Bot_Question_Tracking_2026-03-21_v01.csv")
OUTPUT_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products")
API_BASE = "https://www.metaculus.com/api2/questions"

METACULUS_TOKEN = os.getenv('METACULUS_BOT_API_TOKEN')
RATE_LIMIT_DELAY = 2.0
MAX_RETRIES = 3
BACKOFF_BASE = 5

df_html = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df_html)} questions from HTML extract")
print(f"Columns: {list(df_html.columns)}")
print(f"Auth: {'Token present' if METACULUS_TOKEN else 'NO TOKEN - set METACULUS_BOT_API_TOKEN'}")

Loaded 0 questions from HTML extract
Columns: ['question_number', 'title', 'coverage', 'score', 'question_weight']
Auth: Token present


In [2]:
# API fetch + field extraction functions
def fetch_question_data(question_id):
    """Fetch question data from Metaculus API with exponential backoff."""
    url = f"{API_BASE}/{question_id}/"
    headers = {}
    if METACULUS_TOKEN:
        headers['Authorization'] = f'Token {METACULUS_TOKEN}'
    
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            if response.status_code == 200:
                return response.json()
            if response.status_code == 429:
                wait_time = BACKOFF_BASE * (2 ** attempt)
                print(f" [429, wait {wait_time}s]", end='')
                time.sleep(wait_time)
                continue
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRIES - 1:
                print(f" [ERROR: {e}]", end='')
                return None
            time.sleep(2)
    return None


def extract_fields(data):
    """Extract the 5 API fields we need."""
    question = data.get('question', {})
    q_type = question.get('type', '')
    
    # my_forecast
    my_forecast = 'Not Forecast'
    try:
        my_forecasts = question.get('my_forecasts', {})
        latest = my_forecasts.get('latest', {})
        if latest:
            fv = latest.get('forecast_values', [])
            if q_type == 'binary' and len(fv) == 2:
                my_forecast = f"{fv[1]:.1%}"
            elif q_type == 'numeric':
                means = latest.get('means', [])
                if means:
                    my_forecast = f"{means[0]:.4f}"
                else:
                    my_forecast = json.dumps(fv) if fv else 'Not Forecast'
            elif q_type == 'multiple_choice' and fv:
                options = question.get('options', [])
                if options and len(options) == len(fv):
                    parts = [f"{opt}: {v:.1%}" for opt, v in zip(options, fv)]
                    my_forecast = '; '.join(parts)
                else:
                    my_forecast = json.dumps(fv)
    except Exception:
        pass
    
    # Dates
    open_time = data.get('open_time', '') or ''
    open_date = open_time[:10] if open_time else ''
    
    resolve_time = data.get('actual_resolve_time', '') or ''
    resolution_date = resolve_time[:10] if resolve_time else ''
    
    # Resolution
    resolution = question.get('resolution')
    if resolution is None:
        resolution_value = ''
    elif q_type == 'binary':
        resolution_value = 'Yes' if resolution == 1.0 else ('No' if resolution == 0.0 else str(resolution))
    else:
        resolution_value = str(resolution)
    
    return {
        'my_forecast': my_forecast,
        'open_date': open_date,
        'resolution_date': resolution_date,
        'resolution_value': resolution_value,
        'question_type': q_type,
    }


print("Functions defined")

Functions defined


In [3]:
# Batch fetch all questions
question_numbers = df_html['question_number'].tolist()
api_results = {}
failed = []

print(f"Fetching {len(question_numbers)} questions (~{len(question_numbers) * RATE_LIMIT_DELAY / 60:.0f} min)...\n")

for i, qnum in enumerate(question_numbers, 1):
    if i % 10 == 1 or i == len(question_numbers):
        print(f"[{i}/{len(question_numbers)}] Q{qnum}...", end='')
    else:
        print(f".", end='')
    
    data = fetch_question_data(qnum)
    if data:
        api_results[qnum] = extract_fields(data)
    else:
        failed.append(qnum)
    
    if i < len(question_numbers):
        time.sleep(RATE_LIMIT_DELAY)

forecasted = sum(1 for r in api_results.values() if r['my_forecast'] != 'Not Forecast')
print(f"\n\nFetched: {len(api_results)} | Failed: {len(failed)} | With forecasts: {forecasted}")
if failed:
    print(f"Failed questions: {failed}")

Fetching 0 questions (~0 min)...



Fetched: 0 | Failed: 0 | With forecasts: 0


In [4]:
# Merge API fields into HTML DataFrame
api_df = pd.DataFrame.from_dict(api_results, orient='index')
api_df.index.name = 'question_number'
api_df = api_df.reset_index()

df_merged = df_html.merge(api_df, on='question_number', how='left')

# Reorder columns
col_order = ['question_number', 'title', 'coverage', 'score', 'question_weight',
             'my_forecast', 'open_date', 'resolution_date', 'resolution_value', 'question_type']
df_merged = df_merged[col_order]

# Display
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

print(f"Merged DataFrame: {df_merged.shape}")
print(f"\nWith forecasts: {(df_merged['my_forecast'] != 'Not Forecast').sum()}")
print(f"Resolved: {(df_merged['resolution_value'] != '').sum()}")
print(f"\nType distribution:")
print(df_merged['question_type'].value_counts())
print()
df_merged

KeyError: "['my_forecast', 'open_date', 'resolution_date', 'resolution_value', 'question_type'] not in index"

In [ ]:
# Write output CSV
output_file = OUTPUT_DIR / "Bot_Question_Tracking_API_2026-03-21_v01.csv"
df_merged.to_csv(output_file, index=False)
print(f"Saved: {output_file.name}")
print(f"Rows: {len(df_merged)}")
print(f"Size: {output_file.stat().st_size / 1024:.1f} KB")